In [1]:
import sys
from pathlib import Path

from pandas import DataFrame, Series

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
print(f"Root directory is {ROOT_DIR}")

Root directory is /Users/dcanevarollo/Desktop/Projetos/Unesp/ia/bio-variant-prediction


In [2]:
from bio_var_pred.data.download import download_clinvar
from bio_var_pred.data.annotate import annotate_with_snpeff, index
from bio_var_pred.data.compress import bgzip
from bio_var_pred.utils.paths import INTERIM_DATA_DIR


clinvar_vcf = download_clinvar()

clinvar_vcf = annotate_with_snpeff(
    input_vcf=clinvar_vcf,
    output_vcf=(INTERIM_DATA_DIR / "clinvar_annotated.vcf")
)
clinvar_vcf = bgzip(clinvar_vcf)
index(clinvar_vcf)

print(f"ClinVar data stored in {clinvar_vcf}")

clinvar.vcf.gz already exists. Skipping download.
clinvar_annotated.vcf already annotated. Skipping new annotation.
clinvar_annotated.vcf.gz already compressed. Skipping new compress.
clinvar_annotated.vcf.gz already indexed. Skipping new index.
ClinVar data stored in /Users/dcanevarollo/Desktop/Projetos/Unesp/ia/bio-variant-prediction/data/interim/clinvar_annotated.vcf.gz


In [3]:
from bio_var_pred.data.parse import parse_clinvar


clinvar_df = parse_clinvar(clinvar_vcf)
clinvar_df.head(10)

Parsing ClinVar: 4434137it [00:19, 232277.66it/s]


,id,chrom,pos_genomic,ref,alt,gene,transcript_id,pos_protein,aa_wt,aa_mut,label
0,1164676,1,930165,G,A,SAMD11,ENST00000420190,207,R,Q,0
1,1170208,1,930204,G,A,SAMD11,ENST00000420190,220,R,Q,0
2,1165489,1,930285,G,A,SAMD11,ENST00000420190,247,R,Q,0
3,1170010,1,930314,C,T,SAMD11,ENST00000420190,257,H,Y,0
4,1167937,1,935779,G,A,SAMD11,ENST00000420190,284,G,S,0
5,1167351,1,939429,G,C,SAMD11,ENST00000455979,51,W,C,0
6,1166513,1,942451,T,C,SAMD11,ENST00000455979,169,W,R,0
7,1166588,1,942481,C,T,SAMD11,ENST00000618323,221,P,L,0
8,1167193,1,942577,C,T,SAMD11,ENST00000616016,307,T,I,0
9,1164675,1,942649,C,T,SAMD11,ENST00000616016,331,T,M,0


In [4]:
clinvar_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80492 entries, 0 to 80491
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             80492 non-null  str  
 1   chrom          80492 non-null  str  
 2   pos_genomic    80492 non-null  int64
 3   ref            80492 non-null  str  
 4   alt            80492 non-null  str  
 5   gene           80492 non-null  str  
 6   transcript_id  80492 non-null  str  
 7   pos_protein    80492 non-null  int64
 8   aa_wt          80492 non-null  str  
 9   aa_mut         80492 non-null  str  
 10  label          80492 non-null  int64
dtypes: int64(3), str(8)
memory usage: 9.2 MB


In [5]:
from bio_var_pred.data.annotate import fetch_protein_sequences


transcript_ids: list[str] = clinvar_df["transcript_id"].dropna().unique().tolist()

protein_seq_df = fetch_protein_sequences(transcript_ids, batch_size=50)
protein_seq_df.head(10)

Fetching protein sequences: 100%|██████████| 209/209 [06:06<00:00,  1.75s/it]


,transcript_id,protein_seq
0,ENST00000420190,NaN
1,ENST00000455979,XALLLPRELGPSMAPEDHYRRLVSALSEASTFEDPQRLYHLGLPSH...
2,ENST00000618323,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...
3,ENST00000616016,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...
4,ENST00000379410,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...
5,ENST00000341290,MEPRGGGSSQFSSCPGPASSGDQMQRLLQGPAPRPPGEPPGSPKSP...
6,ENST00000624697,MLAGNEFQVSLSSSMSVSELKAQITQKIGVHAFQQRLAVHPSGVAL...
7,ENST00000379370,MAGRSHPGPLRPLLPLLVVAACVLPGAGGTCPERALERREEEANVV...
8,ENST00000379236,MCVGARRLGRGPCAALLLLGLGLSTVTGLHCVGDTYPSNDRCCHEC...
9,ENST00000379198,MKLLRRAWRRRAALGLGTLALCGAALLYLARCAAEPGDPRAMSGRS...


In [6]:
valid_mean = protein_seq_df["protein_seq"].notna().mean()
print(f"{valid_mean:.2%} of sequences are valid")

97.12% of sequences are valid


In [7]:
protein_seq_df["transcript_id"].is_unique

True

In [8]:
protein_seq_df = protein_seq_df.dropna(subset=["protein_seq"])
protein_seq_df.head()

,transcript_id,protein_seq
1,ENST00000455979,XALLLPRELGPSMAPEDHYRRLVSALSEASTFEDPQRLYHLGLPSH...
2,ENST00000618323,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...
3,ENST00000616016,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...
4,ENST00000379410,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...
5,ENST00000341290,MEPRGGGSSQFSSCPGPASSGDQMQRLLQGPAPRPPGEPPGSPKSP...


In [9]:
clinvar_df = clinvar_df.merge(
    protein_seq_df,
    on="transcript_id",
    how="left",
    validate="many_to_one"
)
clinvar_df = clinvar_df.dropna(subset=["protein_seq"]).reset_index(drop=True)
clinvar_df.head()

,id,chrom,pos_genomic,ref,alt,gene,transcript_id,pos_protein,aa_wt,aa_mut,label,protein_seq
0,1167351,1,939429,G,C,SAMD11,ENST00000455979,51,W,C,0,XALLLPRELGPSMAPEDHYRRLVSALSEASTFEDPQRLYHLGLPSH...
1,1166513,1,942451,T,C,SAMD11,ENST00000455979,169,W,R,0,XALLLPRELGPSMAPEDHYRRLVSALSEASTFEDPQRLYHLGLPSH...
2,1166588,1,942481,C,T,SAMD11,ENST00000618323,221,P,L,0,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...
3,1167193,1,942577,C,T,SAMD11,ENST00000616016,307,T,I,0,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...
4,1164675,1,942649,C,T,SAMD11,ENST00000616016,331,T,M,0,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...


In [10]:
clinvar_df.head()

,id,chrom,pos_genomic,ref,alt,gene,transcript_id,pos_protein,aa_wt,aa_mut,label,protein_seq
0,1167351,1,939429,G,C,SAMD11,ENST00000455979,51,W,C,0,XALLLPRELGPSMAPEDHYRRLVSALSEASTFEDPQRLYHLGLPSH...
1,1166513,1,942451,T,C,SAMD11,ENST00000455979,169,W,R,0,XALLLPRELGPSMAPEDHYRRLVSALSEASTFEDPQRLYHLGLPSH...
2,1166588,1,942481,C,T,SAMD11,ENST00000618323,221,P,L,0,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...
3,1167193,1,942577,C,T,SAMD11,ENST00000616016,307,T,I,0,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...
4,1164675,1,942649,C,T,SAMD11,ENST00000616016,331,T,M,0,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...


In [11]:
clinvar_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 78613 entries, 0 to 78612
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             78613 non-null  str  
 1   chrom          78613 non-null  str  
 2   pos_genomic    78613 non-null  int64
 3   ref            78613 non-null  str  
 4   alt            78613 non-null  str  
 5   gene           78613 non-null  str  
 6   transcript_id  78613 non-null  str  
 7   pos_protein    78613 non-null  int64
 8   aa_wt          78613 non-null  str  
 9   aa_mut         78613 non-null  str  
 10  label          78613 non-null  int64
 11  protein_seq    78613 non-null  str  
dtypes: int64(3), str(9)
memory usage: 118.0 MB


In [12]:
aux_df = clinvar_df.copy()
aux_df["seq_len"] = aux_df["protein_seq"].str.len()

invalid_pos = aux_df[
    aux_df["pos_protein"] > aux_df["seq_len"]
].copy()

print(f"{len(invalid_pos)} problematic variants")
print(f"{100 * len(invalid_pos) / len(aux_df):.2f}% of the dataset")

132 problematic variants
0.17% of the dataset


In [13]:
invalid_pos["transcript_id"].value_counts().head(30)

transcript_id
ENST00000257745    46
ENST00000560249     8
ENST00000425232     8
ENST00000629753     5
ENST00000415075     5
ENST00000330513     5
ENST00000264734     4
ENST00000380149     4
ENST00000274030     3
ENST00000559092     3
ENST00000451102     3
ENST00000617428     3
ENST00000320256     2
ENST00000490416     2
ENST00000217961     2
ENST00000276062     2
ENST00000358075     2
ENST00000616261     1
ENST00000541374     1
ENST00000367762     1
ENST00000402430     1
ENST00000402813     1
ENST00000394701     1
ENST00000336283     1
ENST00000311946     1
ENST00000544175     1
ENST00000354258     1
ENST00000368666     1
ENST00000611959     1
ENST00000472509     1
Name: count, dtype: int64

In [14]:
invalid_pos["gene"].value_counts().head(30)

gene
KMT2E       47
MEPE         8
CPLANE1      8
IKBKB        5
POMT1        5
TGIF1        5
CLDN16       4
ALOXE3       4
USP53        3
SMAD3        3
TK2          3
ARHGEF18     3
ASPRV1       2
NF1          2
STS          2
NDUFB11      2
MAGT1        2
YARS1        1
IL12RB2      1
GORAB        1
ANO7         1
CNGA1        1
AIMP1        1
SRA1         1
NIPAL4       1
BTNL2        1
TAP1         1
CCN6         1
RNASET2      1
TAF6         1
Name: count, dtype: int64

Variants whose annotated amino-acid position exceeded the length of the retrieved protein sequence, or whose wild-type residue did not match the retrieved sequence, were excluded from further analyses.

In [15]:
clinvar_df = clinvar_df[
    clinvar_df["pos_protein"] <= (clinvar_df["protein_seq"].str.len())
]

In [16]:
clinvar_df.head()

,id,chrom,pos_genomic,ref,alt,gene,transcript_id,pos_protein,aa_wt,aa_mut,label,protein_seq
0,1167351,1,939429,G,C,SAMD11,ENST00000455979,51,W,C,0,XALLLPRELGPSMAPEDHYRRLVSALSEASTFEDPQRLYHLGLPSH...
1,1166513,1,942451,T,C,SAMD11,ENST00000455979,169,W,R,0,XALLLPRELGPSMAPEDHYRRLVSALSEASTFEDPQRLYHLGLPSH...
2,1166588,1,942481,C,T,SAMD11,ENST00000618323,221,P,L,0,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...
3,1167193,1,942577,C,T,SAMD11,ENST00000616016,307,T,I,0,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...
4,1164675,1,942649,C,T,SAMD11,ENST00000616016,331,T,M,0,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...


In [17]:
clinvar_df.info()

<class 'pandas.DataFrame'>
Index: 78481 entries, 0 to 78612
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             78481 non-null  str  
 1   chrom          78481 non-null  str  
 2   pos_genomic    78481 non-null  int64
 3   ref            78481 non-null  str  
 4   alt            78481 non-null  str  
 5   gene           78481 non-null  str  
 6   transcript_id  78481 non-null  str  
 7   pos_protein    78481 non-null  int64
 8   aa_wt          78481 non-null  str  
 9   aa_mut         78481 non-null  str  
 10  label          78481 non-null  int64
 11  protein_seq    78481 non-null  str  
dtypes: int64(3), str(9)
memory usage: 118.5 MB


In [18]:
def check_0_based(row):
    try:
        return (
            row["pos_protein"] < len(row["protein_seq"])
            and row["protein_seq"][row["pos_protein"]] == row["aa_wt"]
        )
    except IndexError:
        return False

def check_1_based(row):
    try:
        return (
                1 <= row["pos_protein"] <= len(row["protein_seq"])
                and row["protein_seq"][row["pos_protein"] - 1] == row["aa_wt"]
        )
    except IndexError:
        return False

mask0 = clinvar_df.apply(check_0_based, axis=1)
mask1 = clinvar_df.apply(check_1_based, axis=1)

print("0-based:", mask0.sum())
print("1-based:", mask1.sum())
print("Both:", (mask0 & mask1).sum())
print("None:", (~(mask0 | mask1)).sum())

0-based: 6227
1-based: 76990
Both: 5734
None: 998


998 (≈ 1,28%) of the dataset are inconsistent with the variant position. This is reasonable enough so we can just discard it.

In [19]:
def validate_row(row):
    seq = row["protein_seq"]
    pos = row["pos_protein"]

    if pos < 1:
        return False

    if pos > len(seq):
        return False

    return seq[pos - 1] == row["aa_wt"]

mask = clinvar_df.apply(validate_row, axis=1)
clinvar_df = clinvar_df[mask]

clinvar_df.head(10)

,id,chrom,pos_genomic,ref,alt,gene,transcript_id,pos_protein,aa_wt,aa_mut,label,protein_seq
13,782833,1,966542,G,A,PLEKHN1,ENST00000379410,4,S,N,0,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...
14,782834,1,966543,C,A,PLEKHN1,ENST00000379410,4,S,R,0,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...
15,770481,1,972895,G,A,PLEKHN1,ENST00000379410,346,G,D,0,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...
16,791106,1,973842,G,A,PLEKHN1,ENST00000379410,482,A,T,0,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...
17,1320032,1,976215,A,G,PERM1,ENST00000341290,663,V,A,0,MEPRGGGSSQFSSCPGPASSGDQMQRLLQGPAPRPPGEPPGSPKSP...
18,475283,1,1014042,G,A,ISG15,ENST00000624697,13,S,N,0,MLAGNEFQVSLSSSMSVSELKAQITQKIGVHAFQQRLAVHPSGVAL...
19,402986,1,1014228,G,A,ISG15,ENST00000624697,75,S,N,0,MLAGNEFQVSLSSSMSVSELKAQITQKIGVHAFQQRLAVHPSGVAL...
20,387476,1,1020183,G,C,AGRN,ENST00000379370,4,R,P,0,MAGRSHPGPLRPLLPLLVVAACVLPGAGGTCPERALERREEEANVV...
21,263204,1,1041218,C,T,AGRN,ENST00000379370,258,T,I,0,MAGRSHPGPLRPLLPLLVVAACVLPGAGGTCPERALERREEEANVV...
22,128291,1,1041583,A,G,AGRN,ENST00000379370,353,Q,R,0,MAGRSHPGPLRPLLPLLVVAACVLPGAGGTCPERALERREEEANVV...


In [20]:
clinvar_df.info()

<class 'pandas.DataFrame'>
Index: 76990 entries, 13 to 78612
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             76990 non-null  str  
 1   chrom          76990 non-null  str  
 2   pos_genomic    76990 non-null  int64
 3   ref            76990 non-null  str  
 4   alt            76990 non-null  str  
 5   gene           76990 non-null  str  
 6   transcript_id  76990 non-null  str  
 7   pos_protein    76990 non-null  int64
 8   aa_wt          76990 non-null  str  
 9   aa_mut         76990 non-null  str  
 10  label          76990 non-null  int64
 11  protein_seq    76990 non-null  str  
dtypes: int64(3), str(9)
memory usage: 117.2 MB


In [21]:
from bio_var_pred.data.download import download_gnomad


gnomad_vcf = download_gnomad()
gnomad_vcf = INTERIM_DATA_DIR / gnomad_vcf.name
index(gnomad_vcf)

print(f"gnomAD source downloaded to {gnomad_vcf}")

gnomad.exomes.chr1.vcf.bgz already exists. Skipping download.
gnomad.exomes.chr1.vcf.bgz already indexed. Skipping new index.
gnomAD source downloaded to /Users/dcanevarollo/Desktop/Projetos/Unesp/ia/bio-variant-prediction/data/interim/gnomad.exomes.chr1.vcf.bgz


In [22]:
from bio_var_pred.data.parse import parse_gnomad


gnomad_df = parse_gnomad(gnomad_vcf, chrom="chr1")
gnomad_df.head(10)

Parsing gnomAD: 17671166it [10:43, 27465.44it/s]


,chrom,pos_genomic,ref,alt,af,ac,an
0,1,11994,T,C,NaN,0,0
1,1,12016,G,A,NaN,0,0
2,1,12074,T,C,0.000000,0,4
3,1,12102,G,A,0.000000,0,32
4,1,12106,T,G,0.000000,0,34
5,1,12138,C,A,0.009091,1,110
6,1,12158,C,T,0.000000,0,480
7,1,12165,G,A,0.000000,0,698
8,1,12168,T,G,0.000000,0,626
9,1,12169,A,C,0.000000,0,566


In [23]:
df = clinvar_df.merge(
    gnomad_df,
    on=["chrom", "pos_genomic", "ref", "alt"],
    how="left"
)

In [24]:
df["af"] = df["af"].fillna(0.0)

In [25]:
import numpy as np


df["log_af"] = np.log10(df["af"] + 1e-8)

In [26]:
df.shape

(76990, 16)

In [27]:
df.columns

Index(['id', 'chrom', 'pos_genomic', 'ref', 'alt', 'gene', 'transcript_id',
       'pos_protein', 'aa_wt', 'aa_mut', 'label', 'protein_seq', 'af', 'ac',
       'an', 'log_af'],
      dtype='str')

In [28]:
df.head()

,id,chrom,pos_genomic,ref,alt,gene,transcript_id,pos_protein,aa_wt,aa_mut,label,protein_seq,af,ac,an,log_af
0,782833,1,966542,G,A,PLEKHN1,ENST00000379410,4,S,N,0,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...,0.001402,2041.0,1455774.0,-2.853249
1,782834,1,966543,C,A,PLEKHN1,ENST00000379410,4,S,R,0,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...,0.001402,2041.0,1455976.0,-2.853308
2,770481,1,972895,G,A,PLEKHN1,ENST00000379410,346,G,D,0,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...,0.002800,3938.0,1406536.0,-2.552873
3,791106,1,973842,G,A,PLEKHN1,ENST00000379410,482,A,T,0,MGNSHCVPQAPRRLRASFSRKPSLKGNREDSARMSAGLPGPEAARS...,0.002036,2967.0,1457498.0,-2.691288
4,1320032,1,976215,A,G,PERM1,ENST00000341290,663,V,A,0,MEPRGGGSSQFSSCPGPASSGDQMQRLLQGPAPRPPGEPPGSPKSP...,0.791982,1097406.0,1385646.0,-0.101285


In [29]:
from bio_var_pred.utils.paths import PROCESSED_DATA_DIR

df["gene"] = df["gene"].astype("category")
df.to_parquet(PROCESSED_DATA_DIR / "variants.parquet")